课程目标

掌握蒸馏核心流程、代码实现与优化技巧，完成从教师模型到学生模型的完整知识迁移
前置知识：模型微调、PyTorch基础
课程重难点

    重点：理解知识蒸馏的核心思想——通过教师模型的软标签指导学生模型学习
    重点：掌握温度参数T的作用——控制softmax输出的平滑程度，放大负标签信息
    重点：理解蒸馏损失函数的设计——KL散度损失与交叉熵损失的加权组合
    重点：掌握数据蒸馏的实现流程——使用大模型生成推理数据训练小模型
    难点：区分基于响应、基于特征、基于关系三种蒸馏类型的原理与适用场景
    难点：理解FitNets中间层对齐方法的训练流程与损失函数设计
    难点：区分知识蒸馏与传统微调在目标、方法、应用场景上的本质差异

掌握技能

    能够解释知识蒸馏的核心原理
    能够描述温度参数T对softmax输出分布的影响，理解软标签的优势
    能够说明蒸馏损失函数中KL散度与交叉熵各自的作用
    能够区分离线蒸馏、在线蒸馏、自蒸馏三种方案的特点与适用场景
    能够理解DeepSeek-R1系列蒸馏模型采用的数据蒸馏+SFT训练策略理解 能够解释蒸馏技术在资源受限场景端部署等）



第零部分：为什么需要知识蒸馏
一、现实中对小模型的需求非常大

大模型（如GPT、大ResNet）虽然性能高，但计算量大、存储占用高、推理速度慢，不适合在手机、边缘设备、无人机、车载系统等场景下部署，或者在低带宽、低延迟、低功耗等要求下使用。因此我们希望模型能做到更小、更快、更省电，还能保持较好的准确率。
二、小模型直接训练难以获得高性能

小模型结构简单、容量小，直接训练对小模型训练常常会遇到学不“深”、泛化能力弱等问题。而大模型，由于参数量足够大，在训练过程中学到了很多“隐藏的知识”，我们在谈论知识蒸馏的时候，实际上是希望能够通过一种大模型“带”小模型的方式，实现“走捷径”的效果。
三、提高模型的泛化能力和鲁棒性

对小模型使用硬标签进行训练时，容易出现学得死板、不稳定等现象，而蒸馏使用的是软标签（soft targets）将会更平滑，更有“类间知识”，更容易学出泛化能力和稳定性。这种特性，一些数据稀缺、训练困难的任务中将尤其有效。


第一部分：理论精讲
一、蒸馏技术核心原理
1.知识蒸馏定义

知识蒸馏（Knowledge Distillation）是一种模型压缩技术，通过将一个大型预训练模型（教师模型，Teacher Model）的知识迁移到一个较小的模型（学生模型，Student Model）中，从而在保持模型性能的同时减少模型的参数量和计算量。

知识蒸馏的概念最早是由Geoffrey Hinton等人在《Distilling the Knowledge in a Neural Network》一文中提出。

2.蒸馏技术的基本原理

蒸馏技术的核心在于知识的有效传递与压缩，具体来说，就是通过一个训练有素的教师模型，其凭借复杂的结构和大量的参数捕捉到了数据中的深层次模式和特征。学生模型则通过模仿教师模型的输出来学习这些模式和特征，以期达到相近的表现效果。

蒸馏过程通常包含以下几个关键步骤：

    教师模型的训练

首先需要构建并训练一个高性能的教师模型。这个模型往往拥有庞大的参数量和复杂的网络架构，使其能够深入理解和捕捉输入数据中的复杂模式和特征。

    数据准备

从已经训练好的教师模型中抽取推理数据样本。这一阶段的数据不仅包括原始输入数据，还涵盖了教师模型对这些输入产生的预测结果。这些数据将作为后续训练学生模型的基础资料。

    学生模型的训练

利用教师模型生成的输出作为指导信号（即软目标），对学生模型进行训练。这样做的目的是让学生模型学习到教师模型的知识与经验，尽管它的规模更小、结构更为简单。

    优化与调整

在训练过程中，持续优化和调整学生模型的结构和参数，旨在确保它在维持高效运行的同时，尽可能地逼近教师模型的性能表现。这一步骤可能涉及到多种策略，如调整模型大小、改变网络深度或宽度等，以找到最佳平衡点。

通过上述步骤，蒸馏技术能够在减少计算资源消耗和模型复杂度的情况下，依然保持高水平的任务处理能力。这种方法对于推动机器学习模型向更高效、更实用的方向发展具有重要意义。

3.蒸馏的常见类型与常用方案

    常见类型

    基于响应的知识：这种方法依赖于模型输出层提供的软目标，例如各类别的预测概率。这种方式让学生模型不仅学习教师模型的最终分类结果，还能理解其对不同类别的置信度。

    基于特征的知识：这种类型通过模仿教师模型中间层的特征激活模式来传递知识。这些中间表示能够捕捉输入数据的抽象特征，帮助学生模型学习到更丰富的信息。
    
    基于关系的知识：这里关注的是样本之间或特征之间的交互关系，比如它们的相似性或构成的图结构等。该方法试图让学生模型理解数据点间的复杂关系，而不仅仅是单个实例的信息。



    常用方案

    离线蒸馏：这是一种传统的两步走策略，首先独立训练一个高性能的教师模型，之后使用这个教师模型去指导学生模型的学习。这种方法相对直接，但需要预先准备好一个有效的教师模型。

    在线蒸馏：与离线蒸馏不同，在线蒸馏允许教师和学生模型同时进行训练，并相互学习以提高彼此的表现。这种端到端的联合训练方式（例如深度互学习）可以更灵活地调整两个模型之间的知识转移过程。
    
    自蒸馏：在没有外部教师模型的情况下，也可以在同一模型内部执行知识迁移。例如，可以让深层网络中的高级层指导低级层的学习，这种方法有助于增强模型内部的知识利用效率，特别适合资源有限的情况。


与传统微调的区别

    目标不同：
        知识蒸馏：主要目标是将一个大型预训练模型（教师模型）的知识迁移到一个较小的模型（学生模型）中，从而在保持模型性能的同时减少模型的参数量和计算量。
        传统微调：主要目标是通过在特定任务数据集上进一步训练预训练模型，以提高模型在该任务上的性能。

    方法不同：
        知识蒸馏：通过让学生模型学习教师模型的输出（软标签）和中间层特征，使得学生模型能够在较小的规模下达到与教师模型相近的性能。
        传统微调：通过在特定任务数据集上训练模型，调整模型参数以适应新任务。

    应用场景不同：
        知识蒸馏：适用于需要在资源受限环境中部署模型的场景，如移动设备或嵌入式系统。
        传统微调：适用于需要在特定任务上提高模型性能的场景，如文本分类、图像识别等。

    模型结构不同：
        知识蒸馏：通常涉及两个模型，教师模型和学生模型，学生模型通常比教师模型更小。
        传统微调：通常只涉及一个模型，通过在特定任务数据集上进一步训练该模型。


In [ ]:
# 通过参数变化，感受T对softmax函数输出的影响，假设模型原始输出为[3,4,5]，画柱状图
import numpy as np
import matplotlib.pyplot as plt

def softmax(x,t):
    x=[i/t for i in x]
    return np.exp(x)/np.sum(np.exp(x),axis=0)

xx=[1,2,3]##横坐标
x = [3,4,5]##原始输出向量

print("原始输出：\n")
plt.bar(xx,x)
plt.show()

y = softmax(x,1)
print("原始softmax结果：\n")
plt.bar(xx,y)
plt.show()

print("t=2时softmax结果：\n")
yy=softmax(x,2)
plt.bar(xx,yy)
plt.show()

print("t=0.5时softmax结果：\n")
yyy=softmax(x,0.5)
plt.bar(xx,yyy)
plt.show()

图中展示了蒸馏的训练过程。首先，我们需要准备一个学生模型和一个教师模型。学生模型是一个较小的模型，而教师模型是一个较大的模型，我们假设教师模型已经训练好了，下面主要讨论如何训练学生模型。

我们需要将教师模型的输出，通过一个“高温”操作转换为作为soft label，然后将soft label和hard label结合起来，训练学生模型。如图所示，训练过程中，损失函数应当包括两个部分：一部分是学生模型的损失函数，即学生模型的logits和hard label之间的损失函数，另一部分是soft label和hard label之间的损失函数。具体来说，我们可以使用交叉熵损失函数来计算学生模型的损失函数，然后使用KL散度损失函数来计算soft label和hard label之间的损失函数。

在知识蒸馏（Knowledge Distillation, KD）中，中间层对齐（Intermediate Layer Alignment）是一种扩展传统蒸馏方法的技术，其核心思想是让学生模型不仅模仿教师模型的最终输出，还要模仿其中间层的特征表示。这种方法能更全面地迁移教师模型的“知识”，尤其适用于复杂模型压缩或特征表示学习。

（上面的图中的feature-based knowledge）

中间层对齐有以下特点：

    更丰富的知识迁移：中间层包含局部特征、空间结构等低级信息，而输出层仅提供高级语义。

    缓解容量差距问题：当学生模型过小时，仅模仿输出层可能导致欠拟合，中间层指导能提供更细致的优化目标。

    提升泛化性：中间层特征的对齐可看作一种正则化，防止学生模型过拟合训练标签。

下面介绍一个较为经典的中间层对齐方法，也可以称为特征蒸馏方法，FitNets。

二、蒸馏实战

为了演示deepseek数据蒸馏训练的效果，我们这里使用了开源的数据集中文基于满血DeepSeek-R1蒸馏数据集-110k。


In [ ]:
#!pip install modelscope
!pip install jsonlines

In [ ]:
#数据集下载，更多下载方式，可以参照数据集网页说明
#from modelscope.msdatasets import MsDataset
#ds =  MsDataset.load('Chinese-DeepSeek-R1-Distill-data-110k', subset_name='default', split='train')

我们先看一下这个数据集，这个数据有各类涉及推理，如高考题、数学推理、物理等，我们选择数学推理子集，meta-math/GSM8K_zh，这个数据集有一个唯一解，这样便于我们直观的对比训练的效果。


In [ ]:
import jsonlines
repo_names=set()
with jsonlines.open('Chinese-DeepSeek-R1-Distill-data-110k/distill_r1_110k.jsonl') as reader:
    for obj in reader:
        # print(obj)
        input=obj['input']
        content=obj['content']
        reasoning_content=obj['reasoning_content']
        repo_name=obj['repo_name']
        repo_names.add(repo_name)
    
print(repo_names)

打印一个meta-math/GSM8K_zh的数据。可以看出最终的答案由一个\boxed{}标签包裹，我们只要提出其中的结果即可。而数据集中的reasoing_content字段就是使用deepseek-R1生成的思考过程。

In [ ]:
with jsonlines.open('Chinese-DeepSeek-R1-Distill-data-110k/distill_r1_110k.jsonl')as reader:
    for obj in reader:
        repo_name=obj['repo_name']
        if repo_name=='meta-math/GSM8K_zh':
            input=obj['input']
            content=obj['content']
            reasoning_content=obj['reasoning_content']
            print("input:\n")
            print(input)
            print("content:\n")
            print(content)   
            print("answer:\n")         
            print(content[content.find('boxed'):])
            print("reasoning_content:\n")
            print(reasoning_content)
            break

我们现在的计划是，对大模型直接生成结果、大模型直接训练后生成结果、大模型使用蒸馏数据进行训练后的结果。
(一)直接生成结果

首先，我们让Qwen2.5-1.5B-Instruct直接生成结果，并统计其正确率。

设置一个函数，用来抽取出\boxed{}中的内容。

In [ ]:
import re
def match(text):
    pattern=r'\\boxed\{([^}]*)\}'
    matchs=re.search(pattern,text)
    if matchs:
        return matchs.group(1)
    else:
        return None

In [ ]:
inputs=[]
contents=[]
answers=[]
train_data=[]

#对数据集进行整理，将各要素分别抽取出来，同步生成后续的大模型直接训练的数据
id=0
with jsonlines.open('Chinese-DeepSeek-R1-Distill-data-110k/distill_r1_110k.jsonl')as reader:
    for obj in reader:
        repo_name=obj['repo_name']
        if repo_name=='meta-math/GSM8K_zh':
            id=id+1
            input=obj['input']
            content=obj['content']
            reasoning_content=obj['reasoning_content']
            answer=match(content)
            inputs.append(input)
            contents.append(content)
            answers.append(answer)
            train_data.append({"id":id,"instruction":"请一步步推理，并把最终答案放到 \\boxed{}。\n","input":input,"output":content})


将大模型直接训练的数据，记为base_sft，保存成llamafactory可以直接读取的格式。只存第1001个以后的数据，因为前1000个要作为测试数据。

In [ ]:
import json
with open('lf_data/base_sft.json','w',encoding='utf-8') as f:
    json.dump(train_data[1000:],f,indent=4,ensure_ascii=False)

In [ ]:
#使用vllm部署Qwen2.5-1.5B-Instruct模型，具体操作见run_qwen.sh。同时在代码端与模型进行通联测试。

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:6666/v1",#或改为大模型服务器的地址
    api_key="token-abc123",
)
# prompt="你好"
# completion = client.chat.completions.create(
#   model="Qwen2.5-1.5B",
#   messages=[
#     {"role": "user", "content":prompt}
#   ]
# )
# print(completion)
# print(completion.choices[0].message.content)

#输出结果如下
# ChatCompletion(id='chatcmpl-0806313018e04587aa9d03c7b4875802', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='你好！很高兴见到你。请问有什么我可以帮助你的吗？', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=None)], created=1745421774, model='Qwen2.5-1.5B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=14, prompt_tokens=45, total_tokens=59, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)
# 你好！很高兴见到你。请问有什么我可以帮助你的吗？

对数据集中的前1000个数据，直接进行模型测试，但为了便于处理，我们加上了"请一步步推理，并把最终答案放到 \boxed{}。\n"的指令。这样使直接使用模型进行输出的结果可以以相同的格式输出。

In [ ]:
from tqdm import tqdm
#测试直接生成的方式
count=0
for idx in tqdm(range(len(inputs[:100]))):#有时间可以测1000的
    input=inputs[idx]
    answer=answers[idx]

    for i in range(3):#每组数据测三次，使测试结果更加稳定、准确

        completion = client.chat.completions.create(
        model="Qwen2.5-1.5B",
        messages=[
            {"role": "user", "content":"请一步步推理，并把最终答案放到 \\boxed{}。\n"+input}
        ]
        )
        llm_out=completion.choices[0].message.content
        llm_answer=match(llm_out)
        if llm_answer==answer:
            count=count+1#统计正确个数
            break        
print(count)

# 输出结果如下
# 100%|██████████| 100/100 [03:49<00:00,  2.29s/it]
# 65

直接生成的准确率大约为65%。

(二)大模型直接训练后生成结果

使用（一）中保存下来的训练数据来进行训练，llamafactory的配置llamafactory_train.sh

使用vllm对训练后的模型进行部署并测试，显卡容量充足时，可以直接带lora部署，文件参考run_qwen_base_sft.sh；显卡容量不足时，可以先使用llamafactory进行模型合并，再使用vllm部署，参考llamafactory_exports.sh。。

In [ ]:
from tqdm import tqdm
#测试直接生成的方式
count=0
for idx in tqdm(range(len(inputs[:100]))):
    input=inputs[idx]
    answer=answers[idx]

    for i in range(3):#每组数据测三次，使测试结果更加稳定、准确

        completion = client.chat.completions.create(
        model="Qwen2.5-1.5B-base_sft",
        messages=[
            {"role": "user", "content":"请一步步推理，并把最终答案放到 \\boxed{}。\n"+input}
        ]
        )
        llm_out=completion.choices[0].message.content
        llm_answer=match(llm_out)
        if llm_answer==answer:
            count=count+1#统计正确个数
            break        
print(count)

(三)大模型使用蒸馏数据进行训练

数据集提供了蒸馏数据训练的版本Chinese-DeepSeek-R1-Distill-data-110k-SFT，我们直接使用，下载方式见上文。

In [ ]:
instructions=[]
inputs=[]
outputs=[]
answers=[]
train_data_dis=[]
# instruction": {"_type": "Value"}, "input": {"_type": "Value"}, "output":
#对数据集进行整理，将各要素分别抽取出来，同步生成后续的大模型直接训练的数据
id=0
with jsonlines.open('Chinese-DeepSeek-R1-Distill-data-110k-SFT/distill_r1_110k_sft.jsonl')as reader:
    for obj in reader:
        repo_name=obj['repo_name']
        if repo_name=='meta-math/GSM8K_zh':
            id=id+1
            instruction=obj['instruction']
            input=obj['input']
            output=obj['output']
            # print('instruction')
            # print(instruction)
            # print('input')
            # print(input)
            # print('output')
            # print(output)
            instructions.append(instruction)
            inputs.append(input)
            outputs.append(output)
            answers.append(match(output))
            train_data_dis.append({"id":id,"instruction":instruction,"input":input,"output":output})


把这个数据存成dis_sft.json。

In [ ]:
import json
with open('lf_data/dis_sft.json','w',encoding='utf-8') as f:
    json.dump(train_data_dis[1000:],f,indent=4,ensure_ascii=False)

使用llamafactory进行训练。并以同样的方式进行测试。

In [ ]:
from tqdm import tqdm
#测试直接生成的方式
count=0
for idx in tqdm(range(len(instructions[:100]))):
    input=instructions[idx]
    answer=answers[idx]

    for i in range(3):#每组数据测三次，使测试结果更加稳定、准确

        completion = client.chat.completions.create(
        model="Qwen2.5-1.5B-dis_sft",
        messages=[
            {"role": "user", "content":"请一步步推理，并把最终答案放到 \\boxed{}。\n"+input}
        ]
        )
        llm_out=completion.choices[0].message.content
        # print(llm_out)
        llm_answer=match(llm_out)
        if llm_answer==answer:
            count=count+1#统计正确个数
            break        
print(count)